# 검증 원문 기반 약관 검증 API (/terms/verify-ask) 테스트 노트북

테스트 요청 본문을 **JSON 파일로 관리**하고, 그 파일을 그대로 읽어서 실제 API를 호출합니다.
(Swagger UI에서 JSON을 직접 붙여넣다가 따옴표·줄바꿈 이스케이프가 깨지는 문제를 피하기 위함)

## 사용 전 준비
1. FastAPI 서버를 띄웁니다 (`uvicorn app.main:app`).
2. `REQUEST_JSON_PATH`가 가리키는 JSON 파일을 원하는 내용으로 수정합니다
   (`notebooks/sample_terms_ask_verify_request.json`이 예시로 만들어져 있습니다).
   - `termInfo[].ocrResltKey`는 반드시 `POST /api/v1/documents/parse-di`로 처리 완료된 문서의 해시여야 합니다
     (서버가 이 키로 Azure에 저장된 OCR 결과를 가져옵니다. 없으면 전체 요청이 실패 처리됩니다).
   - 같은 `rqtKey`로 다시 보내면 "이미 처리 중" 또는 "이미 완료(저장된 결과 콜백 재전송)"로 처리됩니다.
     새로 검증하려면 `rqtKey`를 바꾸세요.

In [ ]:
# ── 설정값 ───────────────────────────────────────────────────
from pathlib import Path

API_BASE_URL = "http://localhost:8000"

# 노트북을 notebooks/ 에서 열든 프로젝트 루트에서 열든 동작하도록 경로를 자동으로 잡는다
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REQUEST_JSON_PATH = PROJECT_ROOT / "notebooks" / "sample_terms_ask_verify_request.json"
# ────────────────────────────────────────────────────────────

## 1. 요청 JSON 로드

In [ ]:
import json

import httpx

with open(REQUEST_JSON_PATH, encoding="utf-8") as f:
    request_body = json.load(f)

rqt_key = request_body["rqtKey"]
print(f"rqtKey   : {rqt_key}")
print(f"상품     : {[d['name'] for d in request_body['data']]}")
for t in request_body["termInfo"]:
    print(f"약관 문서: {t['termNm']} → 검증 항목 {[v['itemNm'] for v in t['vrfItem']]}")

## 2. 검증 요청 (`POST /api/v1/terms/verify-ask`)

In [ ]:
response = httpx.post(f"{API_BASE_URL}/api/v1/terms/verify-ask", json=request_body, timeout=30.0)
print(f"HTTP {response.status_code}")
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

## 3. 상태/결과 조회 (`GET /api/v1/terms/verify-ask/{rqtKey}`)

백그라운드 처리 결과를 확인합니다. `status`가 `processing`이면 몇 초 후 다시 실행하세요.
완료되면 `result`에 콜백의 `vrfDataResltJson`과 같은 내용이 나옵니다.

In [ ]:
status_response = httpx.get(f"{API_BASE_URL}/api/v1/terms/verify-ask/{rqt_key}", timeout=30.0)
print(f"HTTP {status_response.status_code}")
status_body = status_response.json()
print(json.dumps(status_body, ensure_ascii=False, indent=2))

완료된 결과를 항목 단위 표로 봅니다 (3번 결과가 완료 상태일 때만).

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 200)

result = status_body.get("result") or {}
rows = [
    {"name": name_result["name"], "termNm": doc["termNm"], "nameMatchRate": doc["nameMatchRate"], **item}
    for name_result in result.get("data", [])
    for doc in name_result["documents"]
    for item in doc["items"]
]
print(f"종합 판정: {result.get('overallResult')} (전체 {result.get('totalCnt')} / 일치 {result.get('matchedCnt')} / "
      f"부분 일치 {result.get('partialMatchCnt')} / 불일치 {result.get('mismatchCnt')})")
pd.DataFrame(rows)

## 4. 콜백 재전송 (`POST /api/v1/terms/verify-ask/{rqtKey}/resend-callback`, 필요할 때만)

In [ ]:
resend_response = httpx.post(f"{API_BASE_URL}/api/v1/terms/verify-ask/{rqt_key}/resend-callback", timeout=30.0)
print(f"HTTP {resend_response.status_code}")
print(json.dumps(resend_response.json(), ensure_ascii=False, indent=2))